## System Project: Cone following and obstacle detection

This notebook introduces a simulation activity for the System Project’s 1:10 scale car. You will build a cone-following policy in Dream Gym, inspect its behaviour, then add obstacles and LiDAR measurements so that you can design and test a stopping response.

The activity follows a practical engineering workflow: define the road and vehicle model, choose the available observations, implement a policy, run the simulation, inspect the results, and revise the policy. The vehicle and sensor settings provide starting values for simulation. Use the results to form questions and tests for your team’s project, then check the required behaviour on the physical car.

Each System Project team will define its own focus. The cone-following and stopping examples show how one part of an autonomous system can be designed, tested, and integrated with sensing and actuation.

**Notebook outline**

- **Setup and imports:** Install the pinned Dream Gym release and select a short or full simulation.
- **1. Define a road with cones:** Construct a route with regular, one-sided, and irregular cone placement.
- **2. Define the vehicle parameters:** Configure a simplified model of the 1:10 scale car and its drive-direction latch.
- **3. Define the cone detection config:** Choose what the simulated camera reports to the policy.
- **4. Make the environment:** Combine the road, vehicle, observations, and initial conditions.
- **5. Cone Following Policy:** Convert detected cone positions into steering and drive commands.
- **6. Simulate the cone following policy:** Inspect the trajectory, time series, and animation.
- **7. Add obstacles to the road:** Place a roadside object and an obstacle in the car’s path.
- **8. Add LiDAR-based observation of obstacles:** Configure a forward-facing obstacle scan.
- **9. Make a new environment with the obstacles:** Combine the existing cone observations with LiDAR measurements.
- **10. Simulate in the obstacle environment:** Observe what happens when the original policy ignores the new sensor.
- **11. Exercise: Add object detection and stopping:** Extend the policy, test its braking response, and investigate which LiDAR beams matter.
- **12. Performance metrics and uncertainty sweeps:** Evaluate the policy across repeated runs and changing conditions.

## Setup and imports


On Colab this installs the pinned course package; locally it does nothing.

In [ ]:
import importlib
import os
from pathlib import Path
import site
import subprocess
import sys
import sysconfig

COURSE_SOURCE_REF = "v0.4.0"
COURSE_REPOSITORY = "https://gitlab.unimelb.edu.au/dream/dream-gym.git"
COLAB_CHECKOUT = Path("/content/dream-gym-current")
IS_COLAB = "google.colab" in sys.modules
package_was_loaded = any(
    name == "dreamgym" or name.startswith("dreamgym.")
    for name in sys.modules
)
setup_diagnostic = None

if IS_COLAB:
    if package_was_loaded:
        setup_diagnostic = "restart the Colab runtime before continuing"
    else:
        try:
            if COLAB_CHECKOUT.exists():
                if not (COLAB_CHECKOUT / ".git").is_dir():
                    raise RuntimeError("unverifiable checkout")
                origin = subprocess.run(
                    ("git", "-C", str(COLAB_CHECKOUT), "remote", "get-url", "origin"),
                    capture_output=True, text=True, check=True,
                ).stdout.strip()
                if origin != COURSE_REPOSITORY:
                    raise RuntimeError("unexpected checkout origin")
                status = subprocess.run(
                    ("git", "-C", str(COLAB_CHECKOUT), "status", "--porcelain"),
                    capture_output=True, text=True, check=True,
                ).stdout.strip()
                if status:
                    raise RuntimeError("checkout has local changes")
                subprocess.run(
                    ("git", "-C", str(COLAB_CHECKOUT), "fetch", "--depth", "1", "origin", COURSE_SOURCE_REF),
                    capture_output=True, text=True, check=True,
                )
                subprocess.run(
                    ("git", "-C", str(COLAB_CHECKOUT), "checkout", "--detach", "FETCH_HEAD"),
                    capture_output=True, text=True, check=True,
                )
            else:
                subprocess.run(
                    ("git", "clone", "--branch", COURSE_SOURCE_REF, "--single-branch", COURSE_REPOSITORY, str(COLAB_CHECKOUT)),
                    capture_output=True, text=True, check=True,
                )
            subprocess.run(
                (sys.executable, "-m", "pip", "install", "-e", "."),
                cwd=COLAB_CHECKOUT, capture_output=True, text=True, check=True,
            )
            site.addsitedir(sysconfig.get_paths()["purelib"])
            os.chdir(COLAB_CHECKOUT)
            importlib.invalidate_caches()
        except (OSError, RuntimeError, subprocess.CalledProcessError):
            setup_diagnostic = "notebook setup failed"

if setup_diagnostic is not None:
    print(setup_diagnostic)


In [ ]:
if setup_diagnostic is not None:
    raise RuntimeError(
        f"Notebook setup did not complete: {setup_diagnostic}"
    )


**Execution mode**

`RUN_MODE` accepts only `smoke` or `full` and defaults to `smoke`. `OUTPUT_ROOT`
is relative to the repository root and all generated artifacts stay below it.

| Property | Units | Allowed values | Default or inheritance | Reason |
| --- | --- | --- | --- | --- |
| `RUN_MODE` | unitless | `smoke` or `full` | default `smoke`; no inheritance | Separates a short path check from explicit long acceptance. |
| `OUTPUT_ROOT` | repository-relative path | `notebook_outputs/cone-detection/` | derived from the selected activity; no inheritance | Contains generated artifacts in the ignored notebook-output root. |
| `MODE.rollout_steps` | steps | smoke `24`; full `600` | selected by `RUN_MODE`; no inheritance | Bounds each later noiseless and noisy rollout. |
| `MODE.animation_frame_step` | transitions/frame | smoke `4`; full `4` | selected by `RUN_MODE`; no inheritance | Selects every fourth later animation transition in either mode. |

The course [notebook setup guide](https://gitlab.unimelb.edu.au/dream/dream-gym/-/blob/v0.4.0/docs/setup_notebooks.md), [Road guide](https://gitlab.unimelb.edu.au/dream/dream-gym/-/blob/v0.4.0/docs/road.md),
[observation guide](https://gitlab.unimelb.edu.au/dream/dream-gym/-/blob/v0.4.0/docs/observations.md), [evaluation utilities guide](https://gitlab.unimelb.edu.au/dream/dream-gym/-/blob/v0.4.0/docs/evaluation_utilities.md),
and [visualization guide](https://gitlab.unimelb.edu.au/dream/dream-gym/-/blob/v0.4.0/docs/visualization.md) describe the public boundaries used by this completed reference.


In [ ]:
RUN_MODE = "full"
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")
OUTPUT_ROOT = Path("notebook_outputs") / "cone-detection"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MODE = {
    "smoke": {"rollout_steps": 24, "animation_frame_step": 4},
    "full": {"rollout_steps": 600, "animation_frame_step": 4},
}[RUN_MODE]


In [ ]:
import dreamgym
dreamgym_source = Path(dreamgym.__file__).resolve()
if IS_COLAB and not dreamgym_source.is_relative_to(COLAB_CHECKOUT.resolve()):
    raise RuntimeError("Dream Gym resolved outside the selected Colab checkout")

from dataclasses import dataclass
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from dreamgym.envs import Road
from dreamgym.evaluation import MetricSet
from dreamgym.evaluation.autonomous_driving import (
    compute_autonomous_driving_metrics,
    metric_by_name,
    simulate_policy,
)
from dreamgym.visualization.autonomous_driving import (
    animate_simulation_trajectory,
    plot_road_from_spec,
    plot_simulation_results,
    save_figure,
    save_road_plot,
    save_trajectory_animation,
)


## 1) Define a road with cones


**Define cone placement**

We use the historical small-car road: a 10 m straight, a 45° left bend of radius 5 m, and an 8 m straight. The lane is 1 m wide. The road's default speed limit is 3.6 km/h (1 m/s), matching the approximate operating speed used by the fixed-drive example later in the notebook.

`cone_paths` defines the two boundaries and their placement profiles. Here blue means left and yellow means right; this is our choice, not a universal colour convention. `lane_line_index` is +1 for the left boundary and −1 for the right boundary of this centred lane.

A `regular` profile places cones every 0.5 m. The right boundary also has a `sparse_displaced` profile with gaps from 0.8–1.2 m and sideways offsets from −0.15–0.15 m. These are physical cone positions, not sensor noise. Distances are in metres.


In [ ]:
CONE_LANE_WIDTH_M = 1.0
CONE_ROAD_PLOT_SEED = 0

CONE_REGULAR_PROFILE = {
    "inter_cone_spacing": {"distribution": "fixed", "value_m": 0.5},
}
CONE_IRREGULAR_PROFILE = {
    "inter_cone_spacing": {
        "distribution": "uniform",
        "lower_bound_m": 0.8,
        "upper_bound_m": 1.2,
    },
    "lateral_displacement": {
        "distribution": "uniform",
        "lower_bound_m": -0.15,
        "upper_bound_m": 0.15,
    },
}
CONE_PATHS = {
    "left_boundary": {
        "placement": {"mode": "lane_line", "lane_line_index": 1},
        "color": "blue",
        "radius_m": 0.08,
        "profiles": {"regular": CONE_REGULAR_PROFILE},
    },
    "right_boundary": {
        "placement": {"mode": "lane_line", "lane_line_index": -1},
        "color": "yellow",
        "radius_m": 0.08,
        "profiles": {
            "regular": CONE_REGULAR_PROFILE,
            "sparse_displaced": CONE_IRREGULAR_PROFILE,
        },
    },
}


**Build the road**

Each element selects its cone profiles using `active_cone_paths`:

1. First straight: regular cones on both sides.
2. Left bend: cones on the left only.
3. Final straight: regular cones on the left, sparse and displaced cones on the right.

An explicit mapping replaces the previous selection; omitting it inherits the previous selection. We write all three mappings explicitly here. The lane width carries forward from the first element.


In [ ]:
CONE_ROAD_SPEC = {
    "defaults": {"speed_limit_kph": 3.6},
    "cone_paths": CONE_PATHS,
    "elements": [
        {
            "geometry": {"type": "straight", "length_m": 10.0},
            "cross_section_width_interpolation": "constant",
            "lanes": {
                "reference_line_role": "lane_center",
                "reference_lane": {
                    "width_m": {"start": CONE_LANE_WIDTH_M, "end": CONE_LANE_WIDTH_M},
                    "surface": "tarmac",
                },
            },
            "active_cone_paths": {
                "left_boundary": "regular",
                "right_boundary": "regular",
            },
        },
        {
            "geometry": {
                "type": "circular_arc",
                "curvature_per_m": 1.0 / 5.0,
                "sweep_angle_deg": 45.0,
            },
            "active_cone_paths": {"left_boundary": "regular"},
        },
        {
            "geometry": {"type": "straight", "length_m": 8.0},
            "active_cone_paths": {
                "left_boundary": "regular",
                "right_boundary": "sparse_displaced",
            },
        },
    ],
}
cone_road = Road(road_spec=CONE_ROAD_SPEC)
resolved_cones = cone_road.resolve_cones(cone_seed=CONE_ROAD_PLOT_SEED)


**Plot and inspect**

The seed makes the irregular placement reproducible. This preview uses one resolved layout; later we will also control the environment's cone seed when running the policy.

**What to notice:** Find the three regions. Which side disappears on the bend? On the final straight, distinguish larger gaps along the boundary from displacement across it.

See the [road cone configuration guide](https://gitlab.unimelb.edu.au/dream/dream-gym/-/blob/v0.4.0/docs/road.md#cones) for additional options.


In [ ]:
cone_road_figure, cone_road_axis = plt.subplots(figsize=(10, 5))
cone_road.plot_road_on_matplotlib_axis(
    cone_road_axis,
    resolved_cones=resolved_cones,
)
cone_road_axis.set_title("Small-car road: regular cones → one side → irregular right side")
cone_road_axis.set_xlabel("world x [m]")
cone_road_axis.set_ylabel("world y [m]")
cone_road_axis.set_aspect("equal", adjustable="box")
plt.show()
plt.close(cone_road_figure)


## 2) Define the vehicle parameters of the 1:10 scale car


**The System Project car in simulation**

In this notebook, we use a simplified bicycle model of the 1:10-scale System Project car to develop and test a cone-following policy.

Our simulation uses a 0.33 m wheelbase and a mass of 3 kg. The distances from the centre of mass to the front and rear axles are 60% and 40% of the wheelbase. We approximate yaw inertia using a rectangular body, and specify the body dimensions used to draw the car.

**Steering and drive response**

In this model, steering requests are limited to ±45°, with a maximum steering rate of 90° per second and zero steering offset. The transition thresholds keep the model in its kinematic regime at the speeds used in this notebook.

Drive commands are normalized between −1 and 1. The configured motor-force gain is 10 N, with a quadratic drag coefficient of 1 kg/m. This initial model uses a linear drive response without a command dead zone. Zero drive removes motor force; it does not instantly stop a moving car.

These settings provide a starting model for simulation; they should not be interpreted as a complete calibration of the physical car.

**Define the vehicle configuration**

The dictionary below defines `bicycle_model_config`, which we will pass to the environment in section 4. Distances use metres, and angles are converted from degrees to radians.


In [ ]:
bicycle_model_config = {
    "axle_geometry": {
        "front_axle_distance_from_center_of_mass_m": 0.60 * 0.33,
        "rear_axle_distance_from_center_of_mass_m": 0.40 * 0.33,
    },
    "mass_properties": {
        "mass_kg": 3.0,
        "yaw_moment_of_inertia_kg_m2": (1.0 / 12.0) * 3.0 * (0.40**2 + 0.25**2),
    },
    "longitudinal_force_model": {
        "normalized_motor_command_force_gain_n": 10.0,
        "quadratic_aerodynamic_drag_coefficient_kg_per_m": 1.0,
        "direction_change_latch": {
            "forward_to_reverse": True,
        },
    },
    "steering": {
        "physical_front_wheel_angle_offset_from_request_rad": 0.0,
        "requested_front_wheel_angle_magnitude_limit_rad": np.deg2rad(45.0),
        "physical_front_wheel_angle_rate_bounds_rad_per_s": {
            "lower": np.deg2rad(-90.0),
            "upper": np.deg2rad(90.0),
        },
    },
    "kinematic_dynamic_transition": {
        "body_longitudinal_speed_range_mps": {
            "lower": 500.0 / 3.6,
            "upper": 600.0 / 3.6,
        },
    },
    "body_geometry": {
        "front_extent_from_center_of_mass_m": (0.60 * 0.33) * 1.5,
        "rear_extent_from_center_of_mass_m": (0.40 * 0.33) * 1.5,
        "width_m": 0.25,
    },
}


**Brake and reverse latch**

The forward-to-reverse direction latch makes the simulated drive command behave more like the System Project car. While the car is moving forward, a negative drive command brakes it and holds it at rest instead of allowing the same held command to begin reversing. To reverse afterward, request neutral while the car is at rest to release the latch, then issue a new negative drive command.

Only the forward-to-reverse latch is enabled here. The reverse-to-forward latch remains disabled.


## 3) Define the cone detection config


**What can the car detect?**

The road configuration determines where cones physically exist. The observation configuration determines which cones the car can detect and how their positions are reported to the policy.

We start with an 80° horizontal field of view and a 4 m forward detection limit. The detector returns up to 24 cones. These are starting simulation settings, not calibrated specifications of the physical camera.

The policy receives only cone detections: forward and left positions in metres relative to the car's body frame, colour IDs, and the number of valid detections. The coordinate and colour arrays have fixed capacity; only the first `count` entries are valid. Remaining entries are padding, not extra cones.

We use the default fixed camera mount at the body-frame origin, pointing forward. Acquisition noise is configured in camera coordinates; the requested output coordinates are in the body frame. With this mount, the two frames coincide.

**Start without position noise**

`CONE_DETECTION_NOISE_STD_M` sets the standard deviation of the position noise in each of the forward and left directions. Start at zero to make the first simulation easier to interpret. A fixed noise seed supports repeatable comparisons when noise is enabled.


In [ ]:
CONE_DETECTION_NOISE_STD_M = 0.0
CONE_NOISE_SEED = 20260811

CONE_OBSERVATION_CONFIG = {
    "cone_detections": {
        "maximum_detection_count": 24,
        "acquisition": {
            "horizontal_field_of_view_deg": 80.0,
            "forward_position_upper_bound_m": 4.0,
            "position_error_in_camera_frame": {
                "forward": {
                    "noise_standard_deviation_m": CONE_DETECTION_NOISE_STD_M,
                },
                "left": {
                    "noise_standard_deviation_m": CONE_DETECTION_NOISE_STD_M,
                },
            },
        },
        "positions_in_body_frame_m": {
            "destination": "observation",
            "scaling_factor": 1.0,
        },
        "color_ids": {"destination": "observation"},
        "count": {"destination": "observation"},
    },
    "fixed_noise_seed": CONE_NOISE_SEED,
}


**Try later: add position noise**

After running the baseline cone-following simulation in section 6, return here and try a small nonnegative value for `CONE_DETECTION_NOISE_STD_M`, such as `0.03` (3 cm). Predict how uncertain cone positions might affect steering, then compare the trajectory with the noiseless result.

Rerun this configuration cell, recreate the environment in section 4, and rerun the policy and simulation cells in sections 5–6. Changing this variable alone does not update an environment that already exists. Keep the road, policy settings and seeds unchanged for the comparison. You can adjust the noise further as you explore; the example value is not a measured sensor specification.

Position noise changes reported cone coordinates, whereas the irregular cone profile in section 1 changes their physical placement. Keep those two sources of variation distinct.


## 4) Make the environment


**Bring the model, road and detector together**

We now combine the road from section 1, the vehicle from section 2 and the observations from section 3.

Start at the beginning of the road, pointing along it and moving at 1 m/s, with a small lateral offset sampled between −0.1 and 0.1 m. Initial lateral velocity, yaw rate and steering use the zero defaults. A fixed reset seed makes the start repeatable.

The simulation advances by 0.05 s per step using RK4 with one integration substep. We use broad termination bounds: planar speed from 0 to 10 m/s and absolute lateral error from the target line up to 20 m. These are simulation stop conditions, not desired operating limits or a lane-keeping success criterion. The simulation horizon will be selected in section 6.

Rendering is disabled here; section 6 will plot and animate the recorded simulation. After changing an earlier configuration, rerun this cell to construct a new environment.


In [ ]:
CONE_RESET_SEED = 0

initial_state_config = {
    "pose": {
        "mode": "road_relative",
        "anchor": "reference_line",
        "progress": {"unit": "metres", "lower": 0.0, "upper": 0.0},
        "lateral_offset_m": {"lower": -0.1, "upper": 0.1},
        "heading_error_rad": {"lower": 0.0, "upper": 0.0},
    },
    "body_motion": {
        "longitudinal_velocity": {
            "source": "absolute",
            "value_mps": {"lower": 1.0, "upper": 1.0},
        },
    },
}
integration_config = {
    "method": "rk4",
    "environment_step_duration_s": 0.05,
    "substep_count": 1,
}
termination_config = {
    "planar_speed_bounds_mps": {"lower": 0.0, "upper": 10.0},
    "absolute_lateral_error_from_target_line_upper_bound_m": 20.0,
}

# Close the previous instance when rerunning this cell.
if "env" in globals():
    env.close()

env = gym.make(
    "dreamgym/autonomous_driving_env",
    road_spec=CONE_ROAD_SPEC,
    bicycle_model_config=bicycle_model_config,
    observation_config=CONE_OBSERVATION_CONFIG,
    initial_state_config=initial_state_config,
    integration_config=integration_config,
    termination_config=termination_config,
    observation_format="dict",
    render_mode=None,
)


**Inspect one observation**

Resetting starts a new episode and returns an observation plus diagnostic information. Display only the valid cone entries below; the rest of each fixed-capacity array is padding.

**What to notice:** Forward and left positions are measured relative to the car, not in world coordinates. Each colour ID belongs to the cone at the same array index. The policy observation contains cone detections only; it does not include the car's true road position. We leave `env` available for the following sections.


In [ ]:
observation, info = env.reset(seed=CONE_RESET_SEED)
cone_count = int(observation["cone_detections_count"][0])
print(f"Valid cone detections: {cone_count}")
display({
    "forward position [m]": observation[
        "cone_detections_forward_positions_in_body_frame_m"
    ][:cone_count],
    "left position [m]": observation[
        "cone_detections_left_positions_in_body_frame_m"
    ][:cone_count],
    "colour ID": observation["cone_detections_color_ids"][:cone_count],
})


## 5) Cone Following Policy


**From detected cones to an action**

A System Project car following a cone-marked route needs to turn its detections into steering and drive commands. Here, we use the detected cone positions and colours to estimate where the route leads and steer towards its centre. The policy uses only the cone observations, without accessing the car's true road position through `info`.

The policy follows five steps:

1. Read the valid detections and separate them by colour: blue marks the left boundary and yellow marks the right boundary on this road.
2. Fit a straight line to each boundary with at least two distinct forward positions. Evaluate each line at the lookahead distance.
3. Average the two boundary estimates to find the centre. If only one boundary is visible, shift its estimate by half the lane width. This allows the car to continue following the route through the bend, where only left-side cones are present.
4. Multiply the estimated centre's lateral offset by a steering gain, limit the steering angle, and convert it to a normalized steering command.
5. Apply a fixed drive command. This policy does not use measured-speed feedback.

Positive lateral offset means left, so a positive steering request turns left. The straight-line fit approximates the nearby boundary: sharp bends, noisy detections or poorly distributed cones can reduce its accuracy. Shifting a single boundary by half the lane width is also approximate when that boundary is angled relative to the car.

If neither boundary provides an estimate, the policy returns zero drive and steering. Zero drive removes motor force but does not instantly stop the car. Consider how your System Project policy should respond when cone detections become unreliable or disappear, and how you would test that response.


In [ ]:
class ConeFollowingPolicy:
    def __init__(self, lane_width_m, lookahead_m, steering_gain_rad_per_m,
                 drive_command, steering_limit_rad):
        # Store the settings chosen in the parameter cell below.
        self.lane_width_m = lane_width_m
        self.lookahead_m = lookahead_m
        self.steering_gain_rad_per_m = steering_gain_rad_per_m
        self.drive_command = drive_command
        self.steering_limit_rad = steering_limit_rad

    def boundary_at_lookahead(self, forward_m, left_m, colour_ids, colour_id):
        # Select this boundary colour, keeping finite positions ahead of the car.
        usable = (
            (colour_ids == colour_id) & (forward_m >= 0.0)
            & np.isfinite(forward_m) & np.isfinite(left_m)
        )
        x, y = forward_m[usable], left_m[usable]
        # A line fit needs at least two different forward positions.
        # None means we cannot estimate this boundary; it is different from zero.
        if len(x) < 2 or np.unique(x).size < 2:
            return None
        # Fit left_position = slope * forward_position + intercept.
        slope, intercept = np.polyfit(x, y, 1)
        # Predict the boundary's lateral position at our chosen distance ahead.
        return slope * self.lookahead_m + intercept

    def compute_action(self, observation, info, terminated, truncated):
        # Match the simulator interface; info is deliberately unused.
        # If the episode has ended, return neutral drive and steering.
        if terminated or truncated:
            return np.zeros(2, dtype=np.float32)

        # Ignore padding: use only the first count entries.
        count = int(observation["cone_detections_count"][0])
        forward_m = observation["cone_detections_forward_positions_in_body_frame_m"][:count]
        left_m = observation["cone_detections_left_positions_in_body_frame_m"][:count]
        colour_ids = observation["cone_detections_color_ids"][:count]

        # Estimate both boundaries at the same lookahead distance.
        # These colour-to-side assignments match the road in section 1.
        left_boundary = self.boundary_at_lookahead(
            forward_m, left_m, colour_ids, Road.CONE_COLOR_TO_ID["blue"]
        )
        right_boundary = self.boundary_at_lookahead(
            forward_m, left_m, colour_ids, Road.CONE_COLOR_TO_ID["yellow"]
        )

        # Both sides visible: aim halfway between them.
        if left_boundary is not None and right_boundary is not None:
            centre_left_m = (left_boundary + right_boundary) / 2.0
        elif left_boundary is not None:
            # The centre is to the right of the left boundary.
            centre_left_m = left_boundary - self.lane_width_m / 2.0
        elif right_boundary is not None:
            # The centre is to the left of the right boundary.
            centre_left_m = right_boundary + self.lane_width_m / 2.0
        else:
            # No usable boundary: remove drive force and request straight steering.
            # This does not instantly stop the car.
            return np.zeros(2, dtype=np.float32)

        # Positive centre offset requests a left turn; negative requests right.
        # Gain converts metres of offset to radians; clip enforces the angle limit.
        steering_rad = np.clip(
            self.steering_gain_rad_per_m * centre_left_m,
            -self.steering_limit_rad, self.steering_limit_rad,
        )
        # The environment expects normalized steering in [-1, 1], not radians.
        steering_command = steering_rad / self.steering_limit_rad
        # Action order is [drive, steering]. Drive is fixed, not speed feedback.
        # float32 matches the environment's action-space dtype.
        return np.array([
            np.clip(self.drive_command, -1.0, 1.0), steering_command,
        ], dtype=np.float32)


**Choose starting values**

Start with a 1 m lookahead, inside the detector's 4 m forward limit, and a steering gain of 0.5 rad per metre of estimated offset. A drive command of 0.1 produces 1 N of motor force with our vehicle configuration; with the configured drag, this roughly balances drag at 1 m/s in straight motion. It is a drive setting, not a speed command.

These are editable starting values rather than an optimized controller. After the simulation in section 6, change one value at a time and compare the path. A longer lookahead or larger gain changes how strongly the car responds to the estimated boundary. Rerun the cell below after changing values so the policy instance uses them.


In [ ]:
CONE_LOOKAHEAD_DISTANCE_M = 1.0
CONE_STEERING_GAIN_RAD_PER_M = 0.5
CONE_DRIVE_COMMAND = 0.1

cone_following_policy = ConeFollowingPolicy(
    lane_width_m=CONE_LANE_WIDTH_M,
    lookahead_m=CONE_LOOKAHEAD_DISTANCE_M,
    steering_gain_rad_per_m=CONE_STEERING_GAIN_RAD_PER_M,
    drive_command=CONE_DRIVE_COMMAND,
    steering_limit_rad=bicycle_model_config["steering"][
        "requested_front_wheel_angle_magnitude_limit_rad"
    ],
)


## 6) Simulate the cone following policy


**Run the simulation**

`simulate_policy` resets `env` with the chosen seed, then repeatedly asks the policy for an action and advances the simulation. It records the results for plotting and stops early if the environment ends the episode.

Smoke mode runs 24 steps (up to 1.2 s): use it to check that everything runs. To inspect the whole route, select `RUN_MODE = "full"` in Setup and imports and rerun the run-settings cell. Full mode allows 600 steps (30 s), enough time to cover this approximately 22 m road at around 1 m/s if tracking is successful.


In [ ]:
cone_results = simulate_policy(
    env,
    MODE["rollout_steps"],
    cone_following_policy,
    seed=CONE_RESET_SEED,
)


**Inspect the trajectory and time series**

Follow the car through the regular cones, the one-sided bend and the irregular final straight. Compare its trajectory with its speed, steering and lateral error over time. Completing a simulation does not by itself show good tracking: our broad termination bounds allow the car to leave the lane.

The speed panel uses km/h and shows the road's 3.6 km/h recommended-speed reference. This equals 1 m/s and is chosen to match the approximate operating point of the fixed-drive example. It remains a reference rather than a speed command: this policy does not use measured-speed feedback.


In [ ]:
cone_result_figures = plot_simulation_results(env, cone_results, should_plot_reward=False)
for figure in cone_result_figures.values():
    display(figure)
    plt.close(figure)


**Watch the car follow the cones**

Use the animation controls to replay the motion and connect it to the plots. The animation shows every fourth simulation step; this changes playback sampling, not the controller's update rate.


In [ ]:
from IPython.display import HTML

cone_animation = animate_simulation_trajectory(
    env,
    cone_results,
    frame_step=MODE["animation_frame_step"],
    figure_title="Following the cone-marked route",
    zoom_width_m=6.0,
    zoom_height_m=6.0,
)
display(HTML(cone_animation.to_jshtml()))
plt.close(cone_animation._fig)


**Try changing the policy**

Predict the effect of changing lookahead or steering gain. Change one parameter in section 5, recreate `cone_following_policy`, then rerun this section. Compare tracking on the bend and the final straight, keeping the seed unchanged.

To investigate position noise, follow the rerun instructions in section 3. Look for changes in steering as well as the path. Identify where the policy struggles and explain what you would change in your System Project controller.


## 7) Add obstacles to the road


**Objects along the route**

Following cones is only part of navigating a route: your System Project car may also encounter objects in its path. Here we add two stationary rectangular obstacles along the route, keeping the road and cones unchanged.

`geometry` defines each rectangle's length and width in metres. `placement` locates its centre using distance along the road's reference line and a lateral offset (positive to the left). `heading_offset_rad` rotates it relative to the road's heading; zero aligns it with the road.

The final straight begins at approximately 13.93 m along the road. At 12 m, on the bend where the right-side cones are absent, a 0.3 m-wide obstacle sits 0.85 m to the right of the reference line. Its nearest edge is approximately 0.2 m outside the lane, leaving room for the car to follow the left-side cones. At 20 m, a centred 0.45 m-wide obstacle blocks the car’s intended path while staying inside the cone boundaries. This creates a sequence of passing a roadside object, then approaching an obstruction that requires a response. We copy the specification first so the baseline road remains available for comparison.


In [ ]:
from copy import deepcopy

ROAD_SPEC_WITH_OBSTACLES = deepcopy(CONE_ROAD_SPEC)
ROAD_SPEC_WITH_OBSTACLES["obstacles"] = [
    {
        "geometry": {"type": "rectangle", "length_m": 0.4, "width_m": 0.3},
        "placement": {
            "reference_line_progress_m": 12.0,
            "lateral_offset_m": -0.85,
            "heading_offset_rad": 0.0,
        },
    },
    {
        "geometry": {"type": "rectangle", "length_m": 0.4, "width_m": 0.45},
        "placement": {
            "reference_line_progress_m": 20.0,
            "lateral_offset_m": 0.0,
            "heading_offset_rad": 0.0,
        },
    },
]
road_with_obstacles = Road(road_spec=ROAD_SPEC_WITH_OBSTACLES)
resolved_cones_with_obstacles = road_with_obstacles.resolve_cones(
    cone_seed=CONE_ROAD_PLOT_SEED,
)


**Inspect the obstacle positions**

The full-road view preserves the context; the closer view makes the small obstacles easier to see. Reusing the cone seed keeps the physical cone placement the same as in section 1.

Which object can the car pass while following the lane, and which blocks its path? Does detecting something nearby always mean the car should stop? A rule based only on the minimum LiDAR range may also respond to roadside objects; we will revisit this distinction when adding sensing and stopping. These obstacles are now part of the road specification, but we have not yet added a sensor or changed the policy. The following sections introduce LiDAR and a separate environment.


In [ ]:
obstacle_figure, obstacle_axes = plt.subplots(1, 2, figsize=(12, 5))
for axis in obstacle_axes:
    road_with_obstacles.plot_road_on_matplotlib_axis(
        axis,
        resolved_cones=resolved_cones_with_obstacles,
        should_render_obstacles=True,
    )
    axis.set_xlabel("world x [m]")
    axis.set_ylabel("world y [m]")
    axis.set_aspect("equal", adjustable="box")

obstacle_axes[0].set_title("Cone-marked road with obstacles")
obstacle_axes[1].set_title("Bend-side object and final obstacle")
obstacle_axes[1].set_xlim(10.5, 19.5)
obstacle_axes[1].set_ylim(-1.5, 7.5)
plt.tight_layout()
plt.show()
plt.close(obstacle_figure)


## 8) Add LiDAR-based observation of obstacles


**Measure distances to obstacles**

The car continues to use cone detections to follow the route. We now add LiDAR measurements of the obstacles from section 7, so the policy can also use information about objects nearby. The cone observations remain unchanged.

We retain the cone observations and add `obstacle_lidar_ranges_m`. Each entry is a distance in metres from the sensor to the first obstacle surface intersected by that beam. A beam with no return reports the maximum range, not zero; a maximum-range reading does not distinguish an obstacle exactly at that limit from no return.

For this example we use 61 beams across a forward-facing 180° field of view, with a 4 m maximum range and a 0.05 m minimum range. The beams are spaced 3° apart: array index 0 points right (−90°), index 30 points forward (0°), and index 60 points left (+90°). The sensor is at the body-frame origin and aligned with the car.

We start with noiseless distance measurements. The settings below define the sensor used in this simulation; measurements closer than 0.05 m are reported as 0.05 m.


In [ ]:
OBSERVATION_CONFIG_WITH_LIDAR = deepcopy(CONE_OBSERVATION_CONFIG)
OBSERVATION_CONFIG_WITH_LIDAR["obstacle_lidar"] = {
    "model": "rplidar_c1",
    "range_encoding": "metres",
    "mounting_pose_in_body_frame": {
        "forward_from_center_of_mass_m": 0.0,
        "left_from_center_of_mass_m": 0.0,
        "heading_offset_from_body_rad": 0.0,
    },
    "acquisition": {
        "beam_count": 61,
        "horizontal_field_of_view_deg": 180.0,
        "minimum_range_m": 0.05,
        "maximum_range_m": 4.0,
        "noise_standard_deviation_m": 0.0,
        "bias_m": 0.0,
        "resolution_m": 0.0,
    },
}


**Predict the scan**

As the car approaches the roadside object on the bend, which beams might detect it? How will those returns differ from the readings as the car approaches the obstacle centred in its path?

The smallest range tells us that an object is nearby, not necessarily that it blocks the route. A stopping rule based only on the minimum range may therefore respond to the roadside object too. Beam direction and the car's width matter when deciding whether an object lies in its path.

This cell defines the additional observations only. Section 9 creates the new environment; section 10 runs the existing cone-following policy, which initially ignores LiDAR. Section 11 introduces a stopping condition. Recreate the obstacle environment after changing these sensor settings.


## 9) Make a new environment with the obstacles


**Create the obstacle environment**

We reuse the vehicle, starting conditions and simulation settings from section 4. Only the road and observation configurations change: this environment contains the obstacles and provides LiDAR ranges alongside cone detections.

We call it `env_with_obstacles` so the original `env` remains available for comparison. Rerun this cell after changing the obstacle or LiDAR configuration.


In [ ]:
# Close the previous obstacle environment when rerunning this cell.
if "env_with_obstacles" in globals():
    env_with_obstacles.close()

env_with_obstacles = gym.make(
    "dreamgym/autonomous_driving_env",
    road_spec=ROAD_SPEC_WITH_OBSTACLES,
    bicycle_model_config=bicycle_model_config,
    observation_config=OBSERVATION_CONFIG_WITH_LIDAR,
    initial_state_config=initial_state_config,
    integration_config=integration_config,
    termination_config=termination_config,
    observation_format="dict",
    render_mode=None,
)


**Check the observations at the start**

Reset with the same seed as the baseline. Cone detections are still available for steering. Both obstacles are initially beyond the 4 m LiDAR range, so the forward reading and minimum range should both be 4 m. The readings will change as the car approaches the obstacles.

For our 61-beam forward-facing scan, the middle entry points straight ahead. The minimum considers all beam directions, so it can also reflect an object beside the car.


In [ ]:
obstacle_observation, obstacle_info = env_with_obstacles.reset(seed=CONE_RESET_SEED)
lidar_ranges_m = obstacle_observation["obstacle_lidar_ranges_m"]
forward_beam_index = len(lidar_ranges_m) // 2

display({
    "valid cone detections": int(obstacle_observation["cone_detections_count"][0]),
    "forward LiDAR range [m]": float(lidar_ranges_m[forward_beam_index]),
    "minimum LiDAR range [m]": float(np.min(lidar_ranges_m)),
})


The environment now supplies obstacle measurements, but `cone_following_policy` still uses only cones. In section 10 we will run that policy in this environment; section 11 will introduce a stopping response.


## 10) Simulate in the obstacle environment


**Run the same policy with obstacles**

The car now receives LiDAR measurements, but its policy still uses only cone detections. Run the simulation to see why adding a sensor does not automatically change the car's behaviour.

Use the same policy, reset seed and simulation horizon as in section 6. Select full mode in Setup and imports and rerun the run-settings cell to reach the obstacles; smoke mode only checks the initial portion. The helper resets `env_with_obstacles` before running.

This run also records the exact measured LiDAR scan at every state for replay in the animation. Recording retains each scan's measured ranges and sensor pose; it does not resample the sensor or reconstruct measurements from obstacle geometry afterward.


In [ ]:
obstacle_results = simulate_policy(
    env_with_obstacles,
    MODE["rollout_steps"],
    cone_following_policy,
    seed=CONE_RESET_SEED,
    should_save_obstacle_lidar=True,
)


**Inspect the trajectory and time series**

Compare these results with section 6. The policy has not changed, so the additional LiDAR observations do not yet affect its actions.

In this simulator, obstacle contact is recorded but does not physically stop or deflect the car. A trajectory can therefore pass through an obstacle. This is not successful obstacle avoidance: the controller must use the available measurements to respond. The speed panel retains the same road-recommended-speed reference explained in section 6.


In [ ]:
obstacle_result_figures = plot_simulation_results(
    env_with_obstacles, obstacle_results, should_plot_reward=False,
)
for figure in obstacle_result_figures.values():
    display(figure)
    plt.close(figure)


**Watch the obstacle encounters**

Watch the car pass the roadside object on the bend and approach the final obstacle. Use the playback controls to inspect whether its steering or drive changes in response to either object.

The blue LiDAR overlay replays the full recorded scan, even though the temporary stopping policy uses only the narrow forward subset. Seeing a roadside return does not therefore mean that the policy used it to request braking.


In [ ]:
from IPython.display import HTML

obstacle_animation = animate_simulation_trajectory(
    env_with_obstacles,
    obstacle_results,
    frame_step=MODE["animation_frame_step"],
    figure_title="Cone following with obstacles",
    zoom_width_m=6.0,
    zoom_height_m=6.0,
    should_render_obstacle_lidar=True,
)
display(HTML(obstacle_animation.to_jshtml()))
plt.close(obstacle_animation._fig)


**What needs to change?**

- Does the car pass the roadside object without leaving the lane?
- Does it respond to the final obstacle, or continue along its cone-following path?
- Where in `compute_action` would you use the LiDAR ranges to change the drive command?

Keep the policy unchanged for this comparison. Section 11 introduces a stopping condition that you can add and test.


## 11) Exercise: Add object detection and stopping to your policy


**Exercise: extend and test your policy**

In this exercise, you will modify the policy in section 5 to use LiDAR ranges, then evaluate its response in section 10. The snippet is a starting point for your implementation.

**Use obstacle distance to change the drive command**

Your car already receives LiDAR ranges. Add a distance-based response to `compute_action` so an object nearby can change its drive command.

In section 5, replace the final `return np.array(...)` block with the snippet below, keeping it inside `compute_action`, after `steering_command` is calculated. The local `drive_command` starts from your chosen setting on every call; it does not overwrite `self.drive_command`.

```python
# Begin with the usual drive command, then check nearby obstacles.
stop_threshold_m = 1.0  # An initial value to investigate, not a guaranteed stopping distance.
drive_command = self.drive_command
lidar_ranges_m = observation["obstacle_lidar_ranges_m"]
if np.min(lidar_ranges_m) < stop_threshold_m:
    drive_command = -1.0  # Request braking through the direction latch.

# Keep the cone-following steering while braking.
return np.array([
    np.clip(drive_command, -1.0, 1.0), steering_command,
], dtype=np.float32)
```

Rerun the policy class and parameter/instantiation cells in section 5, then rerun section 10 in full mode. This modified policy requires the LiDAR observations from `env_with_obstacles`; the original cone-only `env` in section 6 does not supply them. Restore the original return block before using that environment again.

**Does the braking request stop the car in time?**

With the forward-to-reverse direction latch enabled in section 2, a negative drive command while moving forward acts as a braking request. When the car reaches rest, continuing to hold the negative command keeps it stopped instead of allowing it to reverse. Neutral at rest releases the latch; a later negative command can then select reverse.

This distance rule is still not a complete stopping policy. The usual positive drive command returns whenever the minimum range rises above the threshold, so the policy does not remember an earlier decision to stop. The 1 m threshold is an initial value to investigate, not a guaranteed stopping distance. Contact does not physically block motion in this simulator, as discussed in section 10.

- Predict how increasing or decreasing the threshold changes when braking begins, then test one change at a time.
- Does the roadside object cause a response even though the lane remains clear? Minimum range alone does not identify an object in the car's path.

> **Hint:** `np.min(lidar_ranges_m)` considers every beam across the full 180° scan, including objects beside the car. If a roadside object causes unnecessary braking, consider using only the beams near the centre of the array, which point approximately forward. Start with a narrow sector and investigate how widening or narrowing it changes detection of the final obstacle. A fixed forward sector is still only a heuristic, particularly while turning.

> Section 8 describes the LiDAR beam ordering, and Section 9 calculates the index of the forward-facing beam. Consider taking a small slice of `lidar_ranges_m` around that index rather than finding the minimum across the entire scan.

- Inspect speed, drive command and obstacle-contact records as well as the animation. Does speed reach zero before contact? Does positive drive return if the object leaves the scan?
- For your System Project policy, consider how selecting forward-facing beams, accounting for stopping distance and remembering a stop decision could improve the response.

A larger threshold alone is not a guarantee of stopping. Develop and test the response you need, including which beam directions matter for the intended path. The short example above is intended to get that design process started.


## 12) Performance metrics and uncertainty sweeps


The performance-metric and uncertainty-sweep workflows introduced in Upskilling can also be applied to this System Project simulation. Use them to evaluate your policy beyond a single trajectory or animation.

Choose metrics that reflect your team's intended behaviour, such as tracking accuracy, progress and obstacle contact. Then use uncertainty sweeps to investigate how performance changes with conditions such as starting position or sensor noise. Keep the comparison consistent when evaluating different policies.

Use the earlier Upskilling notebooks as a reference for implementing these workflows. Explain what your results show about the policy's strengths, limitations and reliability, and use that evidence to guide your next improvement.
